# 09 残差连接与 LayerNorm

进入 Transformer Encoder 之前，还需要补两个非常重要的结构：

$$
\begin{aligned}
&\text{残差连接（Residual Connection）}\\
&\text{层归一化（LayerNorm）}
\end{aligned}
$$

在 Transformer Encoder 里，一个子层通常不是简单地写成：

$$
\mathbf{y}=\operatorname{Sublayer}(\mathbf{x})
$$

而是会配合残差连接和 LayerNorm。

先给一句核心直觉：

$$
\begin{aligned}
\text{残差连接}&:\text{保留原信息，让网络更容易训练}\\
\operatorname{LayerNorm}&:\text{稳定每个 token 的特征分布}
\end{aligned}
$$

这一节先讲清楚概念和形状，不写代码。

## 1. 为什么进入 Encoder 前要补这两个东西

我们已经学了：

$$
\begin{aligned}
&\operatorname{MultiHeadAttention}\\
&\operatorname{FeedForwardNetwork}\\
&\text{位置编码}
\end{aligned}
$$

但标准 Transformer Encoder Layer 里还会出现：

$$
\operatorname{Add}\quad+\quad\operatorname{Norm}
$$

Add 通常指残差相加。

Norm 通常指 LayerNorm。

如果不先补这两个概念，进入 Encoder 时会看到：

$$
\operatorname{MultiHeadAttention}\rightarrow\operatorname{AddNorm}\rightarrow\operatorname{FFN}\rightarrow\operatorname{AddNorm}
$$

但不知道 Add 和 Norm 为什么在这里。

所以这一节就是给完整 Encoder 铺最后一块地板。

## 2. 先说残差连接是什么

普通子层可以写成：

$$
\mathbf{y}=F(\mathbf{x})
$$

这里 $F$ 可以是 Multi-Head Attention，也可以是 FFN。

残差连接会把原输入 $\mathbf{x}$ 加回去：

$$
\mathbf{y}=\mathbf{x}+F(\mathbf{x})
$$

这就是残差连接最核心的形式。

可以用一句话理解：

$$
\text{子层只需学习在原输入基础上的变化量}
$$

## 3. 残差连接的直觉：保留原始信息

假设某个 token 当前表示是 $\mathbf{x}$。

经过 Attention 或 FFN 后得到：

$$
F(\mathbf{x})
$$

如果直接用 $F(\mathbf{x})$ 替换 $\mathbf{x}$，原来的信息可能会被改得太狠。

残差连接让输出变成：

$$
\mathbf{x}+F(\mathbf{x})
$$

也就是说：

$$
\text{原始信息}+\text{子层学习到的变化量}
$$

这很适合 Transformer。

因为 token 表示在很多层里会不断被改写，残差连接可以帮助信息不要在层层变换中轻易丢失。

## 4. 残差连接的另一个直觉：学习增量

残差连接也可以理解成学习增量。

普通网络像是在学：

$$
\text{普通网络：直接学习最终输出}
$$

残差结构更像是在学：

$$
\text{残差网络：学习在原输入上应补充或修改的增量}
$$

如果某一层暂时没学到有用变化，它可以让 $F(\mathbf{x})$ 接近 0。

这样：

$$
\mathbf{x}+F(\mathbf{x})\approx\mathbf{x}
$$

也就是这层至少可以近似保持输入不变。

这让深层网络更容易训练。

因为模型不必担心每一层都必须做复杂改变。

## 5. 残差连接为什么要求形状一致

残差连接要做加法：

$$
\mathbf{x}+F(\mathbf{x})
$$

能相加的前提是形状一致。

比如输入是：

$$
B\times N\times D
$$

那么子层输出也应该是：

$$
B\times N\times D
$$

这样才能逐元素相加：

$$
(B\times N\times D)+(B\times N\times D)\rightarrow B\times N\times D
$$

这也解释了为什么前面学 FFN 时，最后要从 $d_{\mathrm{ff}}$ 压回 $D$。

如果不压回 $D$，残差连接就加不上了。

## 6. 在 Attention 子层里怎么加残差

假设输入是：

$$
X:B\times N\times D
$$

经过 Multi-Head Attention 后得到：

$$
\operatorname{MHA}(X):B\times N\times D
$$

残差连接就是：

$$
X+\operatorname{MHA}(X)
$$

形状是：

$$
(B\times N\times D)+(B\times N\times D)\rightarrow B\times N\times D
$$

含义是：

$$
\text{原 token 表示}\rightarrow\text{加上 Attention 汇总的上下文信息}
$$

一句话总结两者的关系：

$$
\operatorname{Attention}:\text{负责如何混合信息}\quad+\quad\text{残差连接}:\text{负责让这种混合在深层网络中稳定训练}
$$

也就是说，Attention 解决“信息怎么融合”，残差连接解决“融合之后怎么在深层网络里稳定地传下去”。

## 7. 在 FFN 子层里怎么加残差

FFN 子层也是同样的逻辑。

假设 Attention 子层之后的输入记作 $H$：

$$
H:B\times N\times D
$$

FFN 输出也是：

$$
\operatorname{FFN}(H):B\times N\times D
$$

残差连接：

$$
H+\operatorname{FFN}(H)
$$

形状仍然是：

$$
B\times N\times D
$$

含义是：

$$
\text{FFN 前的表示}\rightarrow\text{加上 FFN 产生的变化量}
$$

## 8. 为什么深层网络喜欢残差连接

Transformer 通常会堆很多层。

如果每一层都完全重写输入表示，训练会很困难。

残差连接提供了一条比较直接的信息通路。

信息和梯度都更容易穿过很多层。

可以先这样理解：

$$
\begin{aligned}
\text{无残差}&:\text{每层完全依赖上一层输出再重新变换}\\
\text{有残差}&:\text{原信息可沿加法通路继续传递}
\end{aligned}
$$

所以残差连接是深层网络能稳定训练的重要原因之一。

## 9. 接下来讲 LayerNorm 是什么

残差连接解决了“保留原信息、帮助深层训练”的问题。

但在深层网络里，还有另一个问题：

$$
\text{层数增加}\rightarrow\text{输出的数值分布可能不断变化}
$$

如果数值太大、太小、分布不稳定，训练就会变得困难。

归一化层的作用就是让数值分布更稳定。

Transformer 里最常用的是 LayerNorm。

它的核心思想是：

$$
\text{每个样本的每个 token}\rightarrow\text{沿特征维度归一化}
$$

## 10. LayerNorm 对谁做归一化

假设输入形状是：

$$
B\times N\times D
$$

其中：

- $B$ 表示 batch size。
- $N$ 表示 token 数量。
- $D$ 表示每个 token 的特征维度。

LayerNorm 通常对最后一维 $D$ 做归一化。

也就是说，对每个 token 的 $D$ 个特征单独计算均值和方差。

可以想成：

- 第 1 个样本的第 1 个 token：对它自己的 $D$ 个特征归一化。
- 第 1 个样本的第 2 个 token：对它自己的 $D$ 个特征归一化。
- 第 2 个样本的第 1 个 token：对它自己的 $D$ 个特征归一化。
- 其余 token 也分别对自己的 $D$ 个特征归一化。

LayerNorm 不需要拿整个 batch 的统计量来归一化。

## 11. 用一个 token 看 LayerNorm

假设某个 token 的向量是 4 维：

$$
\mathbf{x}=[2,4,6,8]
$$

先算这个 token 自己 4 个特征的均值：

$$
\mu=\frac{2+4+6+8}{4}=5
$$

再看每个值和均值的差：

$$
\mathbf{x}-\mu=[-3,-1,1,3]
$$

再用标准差缩放。

简化理解就是让这个 token 的特征分布变得更稳定：

$$
\begin{aligned}
\operatorname{mean}(\hat{\mathbf{x}})&\approx0\\
\operatorname{var}(\hat{\mathbf{x}})&\approx1
\end{aligned}
$$

真实 LayerNorm 还会有可学习的缩放和平移参数，让模型可以调整归一化后的分布。

## 12. LayerNorm 的公式

对一个 token 向量 $\mathbf{x}$，LayerNorm 可以写成：

$$
\operatorname{LayerNorm}(\mathbf{x})=\gamma\frac{\mathbf{x}-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta
$$

这里：

- $\mu$ 是这个 token 各个特征的均值。
- $\sigma^2$ 是这个 token 各个特征的方差。
- $\epsilon$ 是一个很小的数，防止除以 0。
- $\gamma$ 是可学习缩放参数。
- $\beta$ 是可学习平移参数。

先不要被公式吓住。

它的主线就是：

$$
\text{标准化}\rightarrow\text{可学习缩放}\rightarrow\text{可学习平移}
$$

## 13. 为什么 LayerNorm 后形状不变

LayerNorm 改变的是数值分布，不改变形状。

输入是：

$$
B\times N\times D
$$

输出仍然是：

$$
B\times N\times D
$$

因为它只是对每个 token 的 $D$ 个特征做标准化和缩放平移。

不会改变 token 数量。

也不会改变每个 token 的维度。

所以：

$$
\operatorname{LayerNorm}:\text{稳定数值分布，但不改变结构形状}
$$

## 14. LayerNorm 和 BatchNorm 有什么区别

你前面学过 BatchNorm。

BatchNorm 通常依赖 batch 维度上的统计量。

它会看一批样本在某些通道或特征上的均值和方差。

LayerNorm 不一样。

LayerNorm 对每个样本、每个 token 自己的特征维度做归一化。

简单对比：

| 方法 | 主要沿什么方向统计 | 对 batch size 是否敏感 | Transformer 中常见程度 |
|---|---|---|---|
| BatchNorm | batch 维或通道统计 | 比较敏感 | 较少用于标准 Transformer |
| LayerNorm | 单个样本的特征维 | 不依赖 batch 统计 | 非常常见 |

Transformer 处理序列时，batch 大小、序列长度、mask 等情况比较复杂。

LayerNorm 不依赖 batch 统计，因此更适合这类结构。

LayerNorm 把token(样本)的所有维度的特征进行归一化。

BatchNorm 把同一个特征维度的所有样本的数据进行归一化。

## 15. Add & Norm 是什么意思

在 Transformer 图里，经常看到：

$$
\operatorname{Add}\;\&\;\operatorname{Norm}
$$

Add 指残差相加。

Norm 指 LayerNorm。

一种经典写法是：

$$
\operatorname{LayerNorm}(\mathbf{x}+\operatorname{Sublayer}(\mathbf{x}))
$$

读成中文就是：

1. 先让子层处理 $\mathbf{x}$。
2. 再把原来的 $\mathbf{x}$ 加回去。
3. 最后做 LayerNorm。

这里的 Sublayer 可以是：

$$
\operatorname{Sublayer}\in\left\{\operatorname{MultiHeadAttention},\operatorname{FFN}\right\}
$$

## 16. 在 Encoder 里有两次 Add & Norm

一个 Encoder Layer 里通常有两个子层。

第一个子层是 Multi-Head Self-Attention。

第二个子层是 FFN。

所以会有两次 Add & Norm。

可以先写成：

$$
\begin{aligned}
X&\rightarrow\operatorname{MultiHeadAttention}\rightarrow\operatorname{AddNorm}\rightarrow H\\
H&\rightarrow\operatorname{FFN}\rightarrow\operatorname{AddNorm}\rightarrow O
\end{aligned}
$$

更具体一点：

$$
H=\operatorname{LayerNorm}(X+\operatorname{MHA}(X))
$$

$$
O=\operatorname{LayerNorm}(H+\operatorname{FFN}(H))
$$

这就是经典 Post-Norm 形式的直观写法。

## 17. Pre-Norm 和 Post-Norm 先只做概念了解

你以后可能会看到两种写法。

Post-Norm：

$$
\operatorname{LayerNorm}(x+\operatorname{Sublayer}(x))
$$

Pre-Norm：

$$
x+\operatorname{Sublayer}(\operatorname{LayerNorm}(x))
$$

区别是 LayerNorm 放在子层前还是子层后。

入门阶段不要在这里钻太深。

先知道：

- Post-Norm：先经过子层并做残差相加，再做 LayerNorm。
- Pre-Norm：先做 LayerNorm，再经过子层并做残差相加。

现代实现里经常会使用 Pre-Norm 变体，因为训练深层模型时更稳定。

但理解 Encoder 结构时，先掌握 Add、Norm 各自作用即可。

## 18. 残差连接和 LayerNorm 的分工

现在把两者放在一起看。

残差连接解决的是：

$$
\text{残差连接}\rightarrow\left\{\begin{array}{l}\text{保留原始信息}\\\text{帮助深层网络传递信息和梯度}\end{array}\right.
$$

LayerNorm 解决的是：

$$
\operatorname{LayerNorm}\rightarrow\left\{\begin{array}{l}\text{稳定每层输出的数值分布}\\\text{控制每个 token 的特征尺度}\end{array}\right.
$$

所以 Add & Norm 可以理解成：

$$
\text{原信息与子层新信息相加}\rightarrow\operatorname{LayerNorm}\rightarrow\text{稳定表示}
$$

## 19. 常见误解 1：残差连接只是简单多加一次输入

从公式看，残差连接确实是加法。

但它不只是为了多加一点数值。

它改变了网络学习的方式。

普通子层学习完整输出：

普通子层需要直接学习从 $\mathbf{x}$ 到 $\mathbf{y}$ 的完整映射。

残差结构更像学习变化量：

残差结构只需要学习在 $\mathbf{x}$ 的基础上应补充的变化量。

这就是它对深层网络训练很有帮助的原因。

## 20. 常见误解 2：LayerNorm 会改变 token 数量

不会。

LayerNorm 不会改变 token 数量，也不会改变向量维度。

输入：

$$
B\times N\times D
$$

输出还是：

$$
B\times N\times D
$$

它只是对每个 token 的特征数值做标准化和可学习缩放平移。

所以 LayerNorm 改变的是数值分布，不是结构形状。

## 21. 常见误解 3：LayerNorm 和 Softmax 类似

LayerNorm 和 Softmax 都会处理一组数字。

但它们完全不是一回事。

Softmax 的作用是：

$$
\operatorname{Softmax}:\text{分数}\rightarrow\text{非负且总和为 1 的权重分布}
$$

LayerNorm 的作用是：

$$
\operatorname{LayerNorm}:\text{token 特征}\rightarrow\text{均值和方差更稳定的特征}
$$

Softmax 常用于注意力权重。

LayerNorm 常用于稳定隐藏表示。

不要把它们混成同一种归一化。

## 22. 本节小结

这一节先记住：

1. 残差连接的核心形式是 $\mathbf{x}+F(\mathbf{x})$。
2. 残差连接可以保留原始信息，让子层学习增量。
3. 残差连接要求输入和子层输出形状一致。
4. Transformer 子层通常保持 $B\times N\times D$，这让残差相加变得方便。
5. LayerNorm 对每个样本、每个 token 的特征维度做归一化。
6. LayerNorm 不依赖 batch 统计，适合 Transformer。
7. LayerNorm 输入输出形状不变，仍然是 $B\times N\times D$。
8. Add & Norm 通常表示残差相加加 LayerNorm。
9. Encoder Layer 里通常有两次 Add & Norm。
10. Pre-Norm 和 Post-Norm 是 LayerNorm 放置位置不同，先知道概念即可。

## 23. 自测问题

1. 为什么进入 Transformer Encoder 前要先学习残差连接和 LayerNorm？
2. 残差连接的核心公式是什么？
3. 为什么可以把残差连接理解成学习增量？
4. 为什么残差连接要求输入和子层输出形状一致？
5. 在 Attention 子层中，$X+\operatorname{MHA}(X)$ 分别表示什么？
6. LayerNorm 主要解决什么问题？
7. 对 $B\times N\times D$ 的输入，LayerNorm 通常沿哪一维做归一化？
8. LayerNorm 和 BatchNorm 的关键区别是什么？
9. Add & Norm 中的 Add 和 Norm 分别指什么？
10. Encoder Layer 里为什么通常有两次 Add & Norm？
11. Pre-Norm 和 Post-Norm 的区别是什么？
12. LayerNorm 和 Softmax 有什么区别？